# Salary Prediction Project
This notebook combines exploratory data analysis of the initial employee dataset with a synthetic data generation and retraining pipeline to create and save an improved salary prediction model.

## 1. Initial Data Loading and Exploration

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

In [ ]:
df_initial = pd.read_csv("Employees(Employees).csv")
df_initial.head()

In [ ]:
df_initial.info()

In [ ]:
df_initial.shape

In [ ]:
df_initial.isna().sum()

In [ ]:
df_initial.duplicated().sum()

## 2. Data Visualizations (Initial Dataset)

In [ ]:
df_initial['Gender'].value_counts().sort_values(ascending=False).plot(kind='pie', autopct='%1.1f%%')
plt.title("Pie Chart Of Gender Column")
plt.show()

In [ ]:
plt.hist(df_initial["Job Rate"])
plt.title("Histogram Of Job Rate")
plt.xlabel("Rate")
plt.ylabel("Count")
plt.show()

In [ ]:
df_initial.groupby("Department")["Annual Salary"].mean().sort_values(ascending=False).plot(kind="bar")
plt.title("Average Salary Graph By Department")
plt.xlabel("Department")
plt.ylabel("Average Salary")
plt.show()

In [ ]:
# Visualization of average job rate by country (initial dataset)
plt.figure(figsize=(10, 5))
df_initial.groupby("Country")["Job Rate"].mean().sort_values(ascending=False).plot(kind="bar", color="royalblue", edgecolor="black")
plt.title("Average Job Rate by Country (Initial Dataset)", fontsize=14, fontweight="bold")
plt.xlabel("Country", fontsize=12)
plt.ylabel("Average Job Rate", fontsize=12)
plt.xticks(rotation=45)
plt.grid(axis="y", linestyle="--", alpha=0.7)
plt.tight_layout()
plt.show()

In [ ]:
plt.hist(df_initial["Overtime Hours"])
plt.title("Histogram Of Overtime Hours")
plt.xlabel("Overtime Hours")
plt.ylabel("Frequency")
plt.show()

## 3. Realistic Synthetic Data Generation
Since the original dataset is limited in size and missing key features (like Education Level, Employment Type, and specific Skills) that are required for a realistic employee profile, we generate a larger synthetic dataset based on real-world average salary benchmarks.

In [ ]:
np.random.seed(42)
N = 5000

# ── Real-world verified base salaries (USD/year, full-time average) ──────────
COUNTRY_BASE = {
    'United States':        83000,
    'Switzerland':          90000,
    'Australia':            63000,
    'Canada':               55000,
    'Germany':              57000,
    'United Kingdom':       50000,
    'France':               46000,
    'Japan':                38000,
    'Singapore':            52000,
    'Netherlands':          58000,
    'Saudi Arabia':         27000,
    'United Arab Emirates': 30000,
    'Qatar':                32000,
    'Kuwait':               25000,
    'Bahrain':              20000,
    'Oman':                 18000,
    'Turkey':               12000,
    'Brazil':               10000,
    'Mexico':               10000,
    'Malaysia':             12000,
    'South Africa':         11000,
    'Jordan':                8000,
    'Lebanon':               6000,
    'Egypt':                 5500,
    'Morocco':               5000,
    'Indonesia':             5000,
    'Philippines':           5500,
    'India':                 7000,
    'Pakistan':              4000,
    'Nigeria':               4500,
    'Kenya':                 4000,
    'Syria':                 3000,
}

EDU_MULT = {
    'High School':       0.70,
    'Associate Degree':  0.85,
    "Bachelor's Degree": 1.00,
    "Master's Degree":   1.22,
    'PhD':               1.45,
}

EMP_MULT = {
    'Full-Time':  1.00,
    'Part-Time':  0.50,
    'Contract':   1.08,
    'Freelance':  0.90,
    'Internship': 0.30,
}

DEPT_PREMIUM = {
    'AI Development':                  0.45,
    'Machine Learning Engineering':    0.42,
    'AI Research':                     0.40,
    'Generative AI':                   0.38,
    'Data Science':                    0.35,
    'Cybersecurity':                   0.32,
    'Computer Vision':                 0.30,
    'Natural Language Processing':     0.30,
    'Autonomous Systems':              0.28,
    'AI Infrastructure':               0.25,
    'Prompt Engineering':              0.22,
    'Robotics':                        0.28,
    'MLOps':                           0.28,
    'IT':                              0.18,
    'Research/Development':            0.20,
    'Research Center':                 0.18,
    'Product Development':             0.15,
    'Finance':                         0.15,
    'Legal':                           0.12,
    'AI Ethics & Governance':          0.10,
    'Business Development':            0.10,
    'Sales':                           0.08,
    'Account Management':              0.05,
    'Marketing':                       0.05,
    'Operations':                      0.02,
    'Supply Chain':                    0.02,
    'Human Resources':                 0.00,
    'Customer Support':               -0.05,
    'Training':                       -0.05,
    'Professional Training Group':    -0.05,
    'Quality Assurance':              -0.03,
    'Quality Control':                -0.05,
    'Manufacturing':                  -0.08,
    'Manufacturing Admin':            -0.10,
    'Major Mfg Projects':             -0.05,
    'Facilities/Engineering':         -0.03,
    'Green Building':                 -0.03,
    'Environmental Compliance':       -0.05,
    'Environmental Health/Safety':    -0.05,
    'Creative':                        0.00,
}

countries      = list(COUNTRY_BASE.keys())
departments    = list(DEPT_PREMIUM.keys())
edu_levels     = list(EDU_MULT.keys())
emp_types      = list(EMP_MULT.keys())

rows = []
for _ in range(N):
    country    = np.random.choice(countries)
    dept       = np.random.choice(departments)
    edu        = np.random.choice(edu_levels)
    emp_type   = np.random.choice(emp_types)
    years      = int(np.random.randint(0, 31))
    job_rate   = round(float(np.random.uniform(1.0, 5.0)), 1)
    overtime   = int(np.random.randint(0, 400))
    skill_cnt  = int(np.random.randint(1, 9))

    base = COUNTRY_BASE[country]
    seniority_bonus = base * 0.03 * min(years, 30)
    rate_bonus = base * 0.04 * (job_rate - 1)
    hourly = base / 2080
    overtime_bonus = overtime * hourly * 1.5
    skill_bonus = base * 0.02 * skill_cnt
    dept_bonus = base * DEPT_PREMIUM.get(dept, 0)

    salary = (base + seniority_bonus + rate_bonus + overtime_bonus + skill_bonus + dept_bonus)
    salary *= EDU_MULT[edu]
    salary *= EMP_MULT[emp_type]
    salary *= np.random.uniform(0.90, 1.10)
    salary = max(1500, round(salary, 2))

    rows.append({
        'Years':           years,
        'Job Rate':        job_rate,
        'Overtime Hours':  overtime,
        'Skill Count':     skill_cnt,
        'Department':      dept,
        'Country':         country,
        'Education Level': edu,
        'Employment Type': emp_type,
        'Annual Salary':   salary,
    })

df_synthetic = pd.DataFrame(rows)
df_synthetic.head()

## 3.1 Data Visualizations (Synthetic Dataset)
Let's visualize the average job rate across all 32 countries in our generated synthetic dataset.

In [ ]:
# Visualization of average job rate by country in the synthetic dataset
plt.figure(figsize=(14, 6))
df_synthetic.groupby("Country")["Job Rate"].mean().sort_values(ascending=False).plot(kind="bar", color="teal", edgecolor="black")
plt.title("Average Job Rate by Country (Synthetic Dataset)")
plt.xlabel("Country")
plt.ylabel("Average Job Rate (1.0 - 5.0)")
plt.xticks(rotation=90)
plt.grid(axis='y', linestyle='--', alpha=0.7)
plt.tight_layout()
plt.show()

## 4. Production Model Training & Evaluation (Ridge Regression)
We setup a Ridge Regression pipeline using standard scaling for numeric features and one-hot encoding for categorical features. This model will be used in the deployment app.

In [ ]:
features = ['Years', 'Job Rate', 'Overtime Hours', 'Skill Count',
            'Department', 'Country', 'Education Level', 'Employment Type']
X = df_synthetic[features]
y = df_synthetic['Annual Salary']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

num_features = ['Years', 'Job Rate', 'Overtime Hours', 'Skill Count']
cat_features = ['Department', 'Country', 'Education Level', 'Employment Type']

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), num_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), cat_features),
])

pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('regressor', Ridge(alpha=1.0)),
])

pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f"R² Score:  {r2_score(y_test, y_pred):.4f}")
print(f"MAE: ${mean_absolute_error(y_test, y_pred):,.0f}")

In [ ]:
checks = [
    ('United States', 'IT',             "Bachelor's Degree", 'Full-Time', 5, 3.0, 0, 3),
    ('Australia',     'IT',             "Bachelor's Degree", 'Full-Time', 5, 3.0, 0, 3),
    ('India',         'IT',             "Bachelor's Degree", 'Full-Time', 5, 3.0, 0, 3),
    ('Pakistan',      'IT',             "Bachelor's Degree", 'Full-Time', 5, 3.0, 0, 3),
    ('Egypt',         'IT',             "Bachelor's Degree", 'Full-Time', 5, 3.0, 0, 3),
    ('United States', 'AI Development', "Master's Degree",   'Full-Time', 5, 4.0, 0, 6),
]
print("Sanity checks (5 yrs, same profile):")
for c, d, e, et, yr, jr, ot, sk in checks:
    row = pd.DataFrame([{
        'Years': yr, 'Job Rate': jr, 'Overtime Hours': ot, 'Skill Count': sk,
        'Department': d, 'Country': c, 'Education Level': e, 'Employment Type': et
    }])
    pred = pipeline.predict(row)[0]
    print(f"  {c:30s} | {d:25s} | {e:20s} => ${pred:>10,.0f}")

## 5. Model Saving for Web App Deployment

In [16]:
joblib.dump(pipeline, 'linearmodel.pkl')
print("Model saved successfully as linearmodel.pkl!")

Model saved successfully as linearmodel.pkl!
